# 11 Analisis gabungan: sensitivitas metrik, bootstrap, yes/no, leakage, panjang input
Membaca results.json dan prediksi semua run (lihat registry.py) lalu menulis ringkasan analisis A-E, audit panjang input (E7),
dan persiapan studi metrik (E8).

## Setup
Pakai kernel Python dari ~/venvs/dq (venv yang menjalankan semua skrip). Sel ini menyamakan lingkungan dengan bg.sh:
cwd di codeqa_exp/code, pylib_metrics di path, Java untuk METEOR, dan GPU 0.

In [ ]:
import os, sys, json
from pathlib import Path

HOME = Path.home()
EXP = HOME / "inahuy/kacuy/codeqa_exp"
CODE = EXP / "code"
os.chdir(CODE)  # scripts import kc_common / registry from here
for p in (str(CODE), str(EXP / "pylib_metrics")):
    if p not in sys.path:
        sys.path.insert(0, p)
os.environ["PYTHONPATH"] = f"{EXP / 'pylib_metrics'}:{os.environ.get('PYTHONPATH', '')}"
os.environ["PATH"] = f"{HOME / 'envs/kc_tools/bin'}:{os.environ['PATH']}"  # Java for METEOR
os.environ["JAVA_HOME"] = str(HOME / "envs/kc_tools")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("cwd:", os.getcwd(), "| python:", sys.executable)

## analysis.py

In [ ]:
ARGS = []
sys.argv = ['analysis.py']

**Sumber:** code/analysis.py

```text
Analysis A-E and summary.md. Every number is read from results.json / prediction files of the runs.

A metric sensitivity per model and condition; B paired bootstrap (1,000 resamples, seed 42) on
per-example sentence BLEU and F1; C yes/no breakdown with majority baseline; D dev-BLEU vs dev-F1
checkpoint selection; E leakage quintiles and input-length audit.
```

In [ ]:
import json
from collections import Counter

import numpy as np

import kc_common as kc
import registry as reg

In [ ]:
EXP = kc.EXP
test = kc.load_split("test")
train = kc.load_split("train")
N = len(test)
refs = [r["answer"] for r in test]
METRICS = ("BLEU", "BLEU_corpus", "ROUGE_L", "METEOR", "EM", "F1")
notes = []

In [ ]:
def load_metrics(path):
    r = json.loads((EXP / path / "results.json").read_text())
    m = r.get("test", r)  # historical runs keep metrics at top level
    m = dict(m)
    m.setdefault("BLEU_corpus", m.get("corpus_BLEU_diagnostic"))
    return m


def load_per_example(path):
    preds = {int(x["id"]): x["prediction"] for x in kc.read_jsonl(EXP / path / "test_predictions.jsonl")}
    assert len(preds) == N, f"{path}: {len(preds)} predictions"
    p = [preds[i] for i in range(N)]
    per = kc.per_example(p, refs)
    return p, np.array([x["BLEU"] for x in per]), np.array([x["F1"] for x in per]), np.array([x["EM"] for x in per])

In [ ]:
available = {k: v for k, v in reg.RUNS.items() if (EXP / v / "results.json").exists() and (EXP / v / "test_predictions.jsonl").exists()}
for k, v in reg.RUNS.items():
    if k not in available:
        notes.append(f"Missing run for {k[0]} / {k[1]} ({v}); its cells are empty.")
metrics = {k: load_metrics(v) for k, v in available.items()}
per = {k: load_per_example(v) for k, v in available.items()}

In [ ]:
# ---------------- A: metric sensitivity ----------------
table_a = []
for model, label, protocol, primary in reg.MODELS:
    full = metrics.get((model, "FULL"))
    for cond in reg.CONDITIONS:
        m = metrics.get((model, cond))
        if m is None:
            continue
        row = {"model": label, "condition": cond, "protocol": protocol, "primary": primary}
        for k in METRICS:
            v = m.get(k)
            row[k] = v
            if full is not None and cond != "FULL" and v is not None and full.get(k) not in (None, 0):
                row[f"{k}_abs_drop"] = full[k] - v
                row[f"{k}_rel_drop_pct"] = 100 * (full[k] - v) / full[k]
        table_a.append(row)

In [ ]:
# ---------------- B: paired bootstrap ----------------
rng = np.random.default_rng(42)
IDX = rng.integers(0, N, size=(1000, N))  # one shared resample matrix for every comparison
table_b = []
for model, label, _, _ in reg.MODELS:
    if (model, "FULL") not in per:
        continue
    _, fb, ff, _ = per[(model, "FULL")]
    for cond in ("QONLY", "SHUFFLED", "NEIGHBOUR"):
        if (model, cond) not in per:
            continue
        _, ab, af, _ = per[(model, cond)]
        for name, a, f in (("BLEU", ab, fb), ("F1", af, ff)):
            diff = a - f
            boots = diff[IDX].mean(axis=1)
            table_b.append({"model": label, "comparison": f"{cond} - FULL", "metric": name, "mean_diff": float(diff.mean()),
                            "ci95_low": float(np.percentile(boots, 2.5)), "ci95_high": float(np.percentile(boots, 97.5))})

In [ ]:
# ---------------- C: yes/no breakdown ----------------
yn_train = Counter(r["answer"].strip().lower() for r in train if r["answer"].strip().lower() in ("yes", "no"))
majority = yn_train.most_common(1)[0][0]
yn_ids = [i for i, r in enumerate(test) if r["answer"].strip().lower() in ("yes", "no")]
majority_acc = 100 * sum(test[i]["answer"].strip().lower() == majority for i in yn_ids) / len(yn_ids)
table_c = []
for (model, cond), m in metrics.items():
    label = next(l for k, l, _, _ in reg.MODELS if k == model)
    table_c.append({"model": label, "condition": cond, "yes_no_n": m.get("yes_no_n"), "yes_no_correct": m.get("yes_no_correct"),
                    "yes_no_EM": 100 * m["yes_no_correct"] / m["yes_no_n"] if m.get("yes_no_n") else None,
                    "non_yes_no_EM": 100 * m["non_yes_no_correct"] / m["non_yes_no_n"] if m.get("non_yes_no_n") else None})
baseline_c = {"train_majority_label": majority, "train_yes_no_counts": dict(yn_train), "test_yes_no_n": len(yn_ids),
              "majority_baseline_yes_no_accuracy": majority_acc,
              "majority_baseline_overall_EM": 100 * sum(test[i]["answer"].strip().lower() == majority for i in yn_ids) / N}

In [ ]:
# ---------------- D: checkpoint selection ----------------
table_d = []
for cond in ("FULL", "QONLY"):
    path = reg.RUNS.get(("codet5", cond))
    if path and (EXP / path / "results_devF1_selected.json").exists():
        bleu_sel = json.loads((EXP / path / "results.json").read_text())
        f1_sel = json.loads((EXP / path / "results_devF1_selected.json").read_text())
        for name, r in (("dev_BLEU", bleu_sel), ("dev_F1", f1_sel)):
            table_d.append({"model": "CodeT5-small (E1)", "condition": cond, "selection": name, "selected_epoch": r["selected_epoch"],
                            **{k: r["test"][k] for k in ("BLEU", "F1", "EM", "ROUGE_L", "METEOR")}})
gaps_d = {}
for sel in ("dev_BLEU", "dev_F1"):
    rows = {r["condition"]: r for r in table_d if r["selection"] == sel}
    if "FULL" in rows and "QONLY" in rows:
        gaps_d[sel] = {k: rows["FULL"][k] - rows["QONLY"][k] for k in ("BLEU", "F1", "EM", "ROUGE_L", "METEOR")}
s2s_hist = {}
for cond in ("FULL", "QONLY"):
    src = {"FULL": f"{reg.PR}/seq2seq_paper_results/paper_s2s_full_v2_7e0a3fc43cbf",
           "QONLY": f"{reg.PR}/seq2seq_paper_results/paper_s2s_qonly_v2_7159a557a749"}[cond]
    h = json.loads((EXP / src / "history.json").read_text())
    h = h if isinstance(h, list) else h.get("history", h.get("epochs", []))
    def dev(e, k):
        for key in (f"dev_{k}", f"dev/{k}", k):
            if key in e:
                return e[key]
        d = e.get("dev", {})
        return d.get(k) if isinstance(d, dict) else None
    eps = [e for e in h if dev(e, "BLEU") is not None]
    if eps:
        b = max(eps, key=lambda e: dev(e, "BLEU")); f = max(eps, key=lambda e: dev(e, "F1") if dev(e, "F1") is not None else -1)
        s2s_hist[cond] = {"best_dev_BLEU_epoch": b.get("epoch"), "best_dev_F1_epoch": f.get("epoch"),
                          "note": "no per-epoch checkpoints: test of the dev-F1 epoch cannot be computed"}
    else:
        s2s_hist[cond] = {"note": "history.json has no per-epoch dev metrics in a readable form"}

In [ ]:
# ---------------- E: leakage quintiles and length audit ----------------
bins = {x["id"]: x["quintile"] for x in json.loads((EXP / "maps/leakage_bins.json").read_text())["items"]}
leak_items = json.loads((EXP / "maps/leakage_bins.json").read_text())["items"]
table_e = []
for model, label, _, _ in reg.MODELS:
    if (model, "FULL") not in per:
        continue
    _, b, f, em = per[(model, "FULL")]
    for q in range(1, 6):
        ids = [i for i in range(N) if bins[i] == q]
        table_e.append({"model": label, "quintile": q, "n": len(ids), "BLEU": float(b[ids].mean()),
                        "F1": float(f[ids].mean()), "EM": float(em[ids].mean())})
retrieval_baseline = [{"quintile": q, "copy_neighbour_answer_F1": 100 * float(np.mean([x["answer_f1_to_matched"] for x in leak_items if x["quintile"] == q]))}
                      for q in range(1, 6)]
la = json.loads((EXP / "maps/length_audit.json").read_text())
family = {"seq2seq": "seq2seq_ws", "seq2seq_v1": "seq2seq_ws", "codebert": "codebert_bpe", "bart_greedy": "codebert_bpe",
          "bart_beam10": "codebert_bpe", "codet5": "codet5_400"}
table_len = []
for model, label, _, _ in reg.MODELS:
    if (model, "FULL") not in per:
        continue
    _, b, f, em = per[(model, "FULL")]
    fam = family[model]
    t_ids = [x["id"] for x in la["items"] if x[fam]["truncated"]]
    n_ids = [x["id"] for x in la["items"] if not x[fam]["truncated"]]
    table_len.append({"model": label, "tokenizer": fam, "n_truncated": len(t_ids),
                      "BLEU_truncated": float(b[t_ids].mean()) if t_ids else None, "BLEU_not_truncated": float(b[n_ids].mean()),
                      "F1_truncated": float(f[t_ids].mean()) if t_ids else None, "F1_not_truncated": float(f[n_ids].mean())})

In [ ]:
# ---------------- extra: pipeline checks and E4 ----------------
checks = []
for path in sorted((EXP / "runs").glob("*")):
    if path.name.startswith("smoke_") or not (path / "results.json").exists():
        continue
    r = json.loads((path / "results.json").read_text())
    if "agreement_with_historical_predictions" in r:
        a = r["agreement_with_historical_predictions"]
        checks.append({"run": path.name, "identical": a["identical"], "n": a["n"], "pct": a["pct"]})
e4_dev = None
if (EXP / "runs/codebert_t__dev7000_beam10/results.json").exists():
    e4_dev = json.loads((EXP / "runs/codebert_t__dev7000_beam10/results.json").read_text())["dev"]

out = {"A": table_a, "B": table_b, "C": {"rows": table_c, "baseline": baseline_c}, "D": {"rows": table_d, "gaps": gaps_d, "seq2seq": s2s_hist},
       "E": {"leakage": table_e, "retrieval_baseline": retrieval_baseline, "length": table_len, "length_summary": la["summary"]},
       "checks": checks, "E4_codebert_dev7000_beam10": e4_dev, "notes": notes}
kc.write_json(EXP / "results/analysis.json", out)

In [ ]:
# ---------------- summary.md ----------------
def f2(v):
    return "—" if v is None else f"{v:.2f}"

In [ ]:
L = ["# CodeQA Python — context ablations and metric study (seed 42)", "",
     "All numbers below are read from the runs' `results.json` / `test_predictions.jsonl` (per-example scores recomputed with "
     "the shared `codeqa_metrics` evaluator). Test was never used to choose anything.", ""]
L += ["## A. Metric sensitivity (test, 7,000)", "",
      "| Model | Condition | BLEU | BLEU corpus | ROUGE-L | METEOR | EM | F1 | ΔBLEU vs FULL | ΔF1 vs FULL |", "|---|---|---|---|---|---|---|---|---|---|"]
for r in table_a:
    db = f"{r['BLEU_abs_drop']:+.2f} ({r['BLEU_rel_drop_pct']:+.1f}%)" if "BLEU_abs_drop" in r else ""
    df = f"{r['F1_abs_drop']:+.2f} ({r['F1_rel_drop_pct']:+.1f}%)" if "F1_abs_drop" in r else ""
    L.append(f"| {r['model']} | {r['condition']} | {f2(r['BLEU'])} | {f2(r['BLEU_corpus'])} | {f2(r['ROUGE_L'])} | {f2(r['METEOR'])} | {f2(r['EM'])} | {f2(r['F1'])} | {db} | {df} |")
L += ["", "Drops are FULL minus condition (positive = the condition scores lower); relative drop is in % of FULL. "
      "Every metric's absolute and relative drop is in `results/analysis.json` (section A).", ""]
L += ["## B. Paired bootstrap (1,000 resamples, seed 42, one shared resample matrix)", "",
      "| Model | Comparison | Metric | Mean diff | 95% CI |", "|---|---|---|---|---|"]
for r in table_b:
    L.append(f"| {r['model']} | {r['comparison']} | {r['metric']} | {r['mean_diff']:+.2f} | [{r['ci95_low']:+.2f}, {r['ci95_high']:+.2f}] |")
L += ["", "## C. Yes/no breakdown", "",
      f"Majority baseline: the train majority yes/no label is **{baseline_c['train_majority_label']}** "
      f"(train counts {baseline_c['train_yes_no_counts']}); on the {baseline_c['test_yes_no_n']} test yes/no questions it scores "
      f"{baseline_c['majority_baseline_yes_no_accuracy']:.2f}% EM ({baseline_c['majority_baseline_overall_EM']:.2f}% EM over all 7,000).", "",
      "| Model | Condition | yes/no correct | yes/no EM | non-yes/no EM |", "|---|---|---|---|---|"]
for r in sorted(table_c, key=lambda r: (r["model"], reg.CONDITIONS.index(r["condition"]))):
    L.append(f"| {r['model']} | {r['condition']} | {r['yes_no_correct']}/{r['yes_no_n']} | {f2(r['yes_no_EM'])} | {f2(r['non_yes_no_EM'])} |")
L += ["", "## D. Checkpoint selection (dev BLEU vs dev F1)", ""]
if table_d:
    L += ["| Model | Condition | Selected by | Epoch | Test BLEU | Test F1 | Test EM |", "|---|---|---|---|---|---|---|"]
    for r in table_d:
        L.append(f"| {r['model']} | {r['condition']} | {r['selection']} | {r['selected_epoch']} | {f2(r['BLEU'])} | {f2(r['F1'])} | {f2(r['EM'])} |")
    for sel, g in gaps_d.items():
        L.append(f"\nFULL − QONLY gap under {sel} selection: BLEU {g['BLEU']:+.2f}, F1 {g['F1']:+.2f}, EM {g['EM']:+.2f}.")
L.append(f"\nSeq2seq: {json.dumps(s2s_hist)}")
L += ["", "## E. Leakage audit (BM25 similarity quintiles of test code vs training code)", "",
      "| Model (FULL) | Q1 BLEU/F1/EM | Q2 | Q3 | Q4 | Q5 |", "|---|---|---|---|---|---|"]

In [ ]:
for model, label, _, _ in reg.MODELS:
    rows = [r for r in table_e if r["model"] == label]
    if rows:
        L.append(f"| {label} | " + " | ".join(f"{r['BLEU']:.1f} / {r['F1']:.1f} / {r['EM']:.1f}" for r in rows) + " |")
L.append("| Copy the matched training answer (F1) | " + " | ".join(f"{r['copy_neighbour_answer_F1']:.1f}" for r in retrieval_baseline) + " |")
L += ["", "### Input-length audit (test)", "", "| Tokenizer | Truncated | Code tokens lost (total) | Median lost if truncated |", "|---|---|---|---|"]
for fam, s in la["summary"].items():
    L.append(f"| {fam} | {s['n_truncated']} ({s['pct_truncated']:.2f}%) | {s['code_tokens_lost_total']} | {s['code_tokens_lost_median_if_truncated']} |")
L += ["", "| Model (FULL) | Truncated n | BLEU truncated / not | F1 truncated / not |", "|---|---|---|---|"]
for r in table_len:
    L.append(f"| {r['model']} | {r['n_truncated']} | {f2(r['BLEU_truncated'])} / {f2(r['BLEU_not_truncated'])} | {f2(r['F1_truncated'])} / {f2(r['F1_not_truncated'])} |")
L += ["", "## Pipeline checks", "", "| Re-decode run | Identical to historical predictions |", "|---|---|"]
for c in checks:
    L.append(f"| {c['run']} | {c['identical']}/{c['n']} ({c['pct']:.2f}%) |")
if e4_dev:
    L += ["", f"E4 CodeBERT-Transformer, full 7,000 dev, beam 10: BLEU {e4_dev['BLEU']:.2f}, F1 {e4_dev['F1']:.2f}, EM {e4_dev['EM']:.2f} "
          f"(historical selection used the 1,000-example dev subset)."]
if notes:
    L += ["", "## Missing pieces", ""] + [f"- {n}" for n in notes]
(EXP / "results").mkdir(exist_ok=True)
(EXP / "results/summary_tables.md").write_text("\n".join(L) + "\n", encoding="utf-8")

run = kc.wandb_init(EXP / "results", "analysis_A_to_E", "ANALYSIS", ["all", "ANALYSIS", "none", "seed42"],
                    {"experiment": "ANALYSIS", "bootstrap_resamples": 1000, "seed": 42}, "analysis")

In [ ]:
import wandb  # noqa: E402

In [ ]:
def table(rows):
    cols = sorted({k for r in rows for k in r}, key=lambda k: list(rows[0]).index(k) if k in rows[0] else 999)
    t = wandb.Table(columns=cols)
    for r in rows:
        t.add_data(*[r.get(c) if not isinstance(r.get(c), (dict, list)) else json.dumps(r.get(c)) for c in cols])
    return t

In [ ]:
logs = {"A_metric_sensitivity": table(table_a), "C_yes_no": table(table_c)}
if table_b: logs["B_bootstrap"] = table(table_b)
if table_d: logs["D_checkpoint_selection"] = table(table_d)
if table_e: logs["E_leakage_quintiles"] = table(table_e)
if table_len: logs["E_length_audit"] = table(table_len)
run.log(logs)
art = wandb.Artifact("analysis-outputs", type="results")
art.add_file(str(EXP / "results/analysis.json")); art.add_file(str(EXP / "results/summary_tables.md"))
run.log_artifact(art)
run.finish()
print("DONE analysis", len(table_a), "A rows,", len(table_b), "bootstrap rows,", len(notes), "notes", flush=True)

## length_audit.py

In [ ]:
ARGS = []
sys.argv = ['length_audit.py']

**Sumber:** code/length_audit.py

```text
E7: input-length audit on test, per tokenizer family, mirroring each FULL run's source construction.

seq2seq_ws:   ["[CLS]"] + question.split() + ["[SEP]"] + code.split(), lowercase, limit 400
codebert_bpe: [CLS] + (tok(question) + [SEP] + tok(code))[:398] + [SEP]   (CodeBERT-Transformer and CodeBERT-BART)
codet5_400:   tokenizer("question: {q}
code: {c}") with limit 400 (E1 runs)
codet5_512:   same template with limit 512 (old non-compliant CodeT5 run, reference only)
Truncation always cuts the end, so every lost token is a code token.
```

In [ ]:
from transformers import AutoTokenizer, RobertaTokenizer

import kc_common as kc

In [ ]:
test = kc.load_split("test")
cb = RobertaTokenizer.from_pretrained("microsoft/codebert-base", revision="3b0952feddeffad0063f274080e3c23d75e7eb39")
t5 = AutoTokenizer.from_pretrained("Salesforce/codet5-small")

items = []
for r in test:
    q, c = r["question"], r["code"]
    ws_len = 2 + len(q.split()) + len(c.split())
    ws_code = len(c.split())
    cb_q, cb_c = cb.tokenize(q), cb.tokenize(c)
    cb_len = 2 + len(cb_q) + 1 + len(cb_c)
    t5_full = len(t5(kc.codet5_source(q, c, "full"))["input_ids"])
    t5_prefix = len(t5(kc.codet5_source(q, c, "qonly"))["input_ids"])
    t5_code = t5_full - t5_prefix
    entry = {"id": r["id"]}
    for name, length, code_len, limit in (("seq2seq_ws", ws_len, ws_code, 400), ("codebert_bpe", cb_len, len(cb_c), 400),
                                          ("codet5_400", t5_full, t5_code, 400), ("codet5_512", t5_full, t5_code, 512)):
        lost = max(0, length - limit)
        entry[name] = {"source_len": length, "code_tokens": code_len, "truncated": lost > 0,
                       "code_tokens_lost": min(lost, code_len), "code_fraction_lost": (min(lost, code_len) / code_len) if code_len else 0.0}
    items.append(entry)

summary = {}
for name in ("seq2seq_ws", "codebert_bpe", "codet5_400", "codet5_512"):
    trunc = [x[name] for x in items if x[name]["truncated"]]
    lost = sorted(x["code_tokens_lost"] for x in trunc)
    summary[name] = {"n_truncated": len(trunc), "pct_truncated": 100 * len(trunc) / len(items),
                     "code_tokens_lost_total": sum(lost),
                     "code_tokens_lost_median_if_truncated": lost[len(lost) // 2] if lost else 0,
                     "mean_code_fraction_lost_if_truncated": (sum(x["code_fraction_lost"] for x in trunc) / len(trunc)) if trunc else 0.0}
    print(name, summary[name], flush=True)
kc.write_json(kc.EXP / "maps/length_audit.json", {"summary": summary, "items": items})

## metric_study.py

In [ ]:
ARGS = []
sys.argv = ['metric_study.py']

**Sumber:** code/metric_study.py

```text
E8: metric-study preparation on 300 stratified test IDs. No labelling is done here.

For every condition in conditions.json: per-example sentence BLEU, EM, F1, ROUGE-L, METEOR (official
evaluators), corpus BLEU per condition (diagnostic), BERTScore and PEDANTS when importable (errors are
logged and the rest continues). Exports a blind labelling sheet and a separate key.
```

In [ ]:
import csv
import json
import random
import sys
import traceback

import kc_common as kc

In [ ]:
OUT = kc.EXP / "metric_study"
OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(kc.EXP / "pylib_metrics"))
errors = []

ids = json.loads((kc.EXP / "maps/metric_study_ids.json").read_text())["ids"]
test = {r["id"]: r for r in kc.load_split("test")}

In [ ]:
import registry as reg  # noqa: E402

In [ ]:
registry = []
for (model, condition), path in reg.RUNS.items():
    if (kc.EXP / path / "test_predictions.jsonl").exists():
        registry.append({"model": model, "condition": condition, "run_tag": path.split("/")[-1],
                         "predictions": f"{path}/test_predictions.jsonl"})
    else:
        errors.append(f"{model}/{condition}: {path}/test_predictions.jsonl not found; condition skipped")

rows = []  # one per (id, condition)
for cond in registry:
    preds = {int(r["id"]): r["prediction"] for r in kc.read_jsonl(kc.EXP / cond["predictions"])}
    missing = [i for i in ids if i not in preds]
    if missing:
        errors.append(f"{cond['run_tag']}: {len(missing)} study ids missing; condition skipped")
        continue
    for i in ids:
        rows.append({"id": i, "model": cond["model"], "condition": cond["condition"], "run_tag": cond["run_tag"],
                     "question": test[i]["question"], "reference": test[i]["answer"], "prediction": preds[i]})
print(f"{len(rows)} rows from {len(registry)} conditions", flush=True)

predictions = [r["prediction"] for r in rows]
references = [r["reference"] for r in rows]
for r, s in zip(rows, kc.per_example(predictions, references)):
    r.update(BLEU=s["BLEU"], F1=s["F1"], EM=s["EM"])

try:
    rouge = kc.codeqa_metrics._load("rouge/rouge.py", str(kc.REPO)).Rouge()
    _, per = rouge.compute_score({i: [t] for i, t in enumerate(references)}, {i: [t] for i, t in enumerate(predictions)})
    for k, r in enumerate(rows):
        r["ROUGE_L"] = 100.0 * float(per[k])
except Exception:
    errors.append("ROUGE-L: " + traceback.format_exc(limit=2))

In [ ]:
try:
    path = kc.REPO / "source/codeBERT/code/eval/meteor/meteor.py"
    text = path.read_text(encoding="utf-8").replace("['java', '-jar', '-Xmx{}'.format(mem), METEOR_JAR,",
                                                    "['java', '-Xmx{}'.format(mem), '-jar', METEOR_JAR,")
    namespace = {"__file__": str(path)}
    exec(compile(text, str(path), "exec"), namespace)
    meteor = namespace["Meteor"]()
    _, per = meteor.compute_score({i: [t] for i, t in enumerate(references)}, {i: [t] for i, t in enumerate(predictions)})
    meteor.close()
    for r, v in zip(rows, per):
        r["METEOR"] = 100.0 * v
except Exception:
    errors.append("METEOR: " + traceback.format_exc(limit=2))

try:
    import bert_score
    _, _, f = bert_score.score(predictions, references, lang="en", batch_size=128, verbose=False)
    for r, v in zip(rows, f.tolist()):
        r["BERTScore_F1"] = 100.0 * v
    errors.append(f"info: BERTScore {bert_score.__version__}, default English model (roberta-large), no baseline rescaling")
except Exception:
    errors.append("BERTScore: " + traceback.format_exc(limit=3))

try:
    from qa_metrics.pedant import PEDANT
    pedant = PEDANT()
    for r in rows:
        r["PEDANTS"] = float(pedant.get_score(r["reference"], r["prediction"], r["question"]))
except Exception:
    errors.append("PEDANTS: " + traceback.format_exc(limit=3))

# Corpus BLEU per condition (diagnostic), computed on the 300 study items.
bleu = kc.codeqa_metrics._load("bleu/google_bleu.py", str(kc.REPO))
summary = {}

In [ ]:
for cond in registry:
    sub = [r for r in rows if r["run_tag"] == cond["run_tag"]]
    if not sub:
        continue
    corpus, _, _ = bleu.corpus_bleu({k: [r["prediction"]] for k, r in enumerate(sub)}, {k: [r["reference"]] for k, r in enumerate(sub)})
    metrics = {m: sum(r[m] for r in sub) / len(sub) for m in ("BLEU", "F1", "EM", "ROUGE_L", "METEOR", "BERTScore_F1", "PEDANTS")
               if all(m in r for r in sub)}
    summary[cond["run_tag"]] = {"model": cond["model"], "condition": cond["condition"], "n": len(sub),
                                "BLEU_corpus": 100.0 * corpus, **metrics}

metric_cols = [c for c in ("BLEU", "F1", "EM", "ROUGE_L", "METEOR", "BERTScore_F1", "PEDANTS") if all(c in r for r in rows)]
with open(OUT / "scores.csv", "w", newline="", encoding="utf-8") as stream:
    w = csv.writer(stream)
    w.writerow(["id", "model", "condition", "run_tag", "prediction", "reference", *metric_cols])
    for r in rows:
        w.writerow([r["id"], r["model"], r["condition"], r["run_tag"], r["prediction"], r["reference"], *[r[c] for c in metric_cols]])

# Blind labelling sheet: shuffled (seed 42), model/condition hidden; code shown is the ORIGINAL test code.
order = list(range(len(rows)))
random.Random(42).shuffle(order)
with open(OUT / "labeling_sheet.csv", "w", newline="", encoding="utf-8") as sheet, \
        open(OUT / "labeling_key.csv", "w", newline="", encoding="utf-8") as key:
    ws, wk = csv.writer(sheet), csv.writer(key)
    ws.writerow(["item_id", "question", "code", "reference", "prediction", "label_annotator", "notes"])
    wk.writerow(["item_id", "id", "model", "condition", "run_tag"])
    for item, k in enumerate(order, start=1):
        r = rows[k]
        ws.writerow([item, r["question"], test[r["id"]]["code"], r["reference"], r["prediction"], "", ""])
        wk.writerow([item, r["id"], r["model"], r["condition"], r["run_tag"]])

kc.write_json(OUT / "summary.json", {"n_ids": len(ids), "n_rows": len(rows), "metrics": metric_cols,
                                      "per_condition": summary, "errors_and_info": errors})
for e in errors:
    print(e.splitlines()[0] if e else e, flush=True)

In [ ]:
run = kc.wandb_init(OUT, "e8_metric_study", "E8", ["all", "METRIC_STUDY", "none", "seed42"],
                    {"experiment": "E8", "n_ids": len(ids), "conditions": [c["run_tag"] for c in registry],
                     "metrics": metric_cols, "errors": errors}, "analysis")

In [ ]:
import wandb  # noqa: E402

In [ ]:
table = wandb.Table(columns=["run_tag", "model", "condition", "n", "BLEU_corpus", *metric_cols])
for tag, s in summary.items():
    table.add_data(tag, s["model"], s["condition"], s["n"], s["BLEU_corpus"], *[s.get(c) for c in metric_cols])
run.log({"e8/per_condition_means": table})
artifact = wandb.Artifact("e8_metric_study-outputs", type="results")
for name in ("scores.csv", "summary.json", "labeling_sheet.csv", "labeling_key.csv"):
    artifact.add_file(str(OUT / name))
run.log_artifact(artifact)
run.finish()
print("DONE metric study", len(rows), "rows", flush=True)